# 03 --  Feature Engineering

## Concept
Feature engineering transforms domain knowledge into informative ML features. Good features make even simple models perform well.

## Mathematical Intuition
- **Ratio features**: x_1/x_2 --  captures relative relationships
- **Aggregate features**: mean, max, min of related entities
- **Geospatial features**: Haversine distance, clustering
- **Polynomial features**: x_1^2, x_1xx_2 --  capture non-linear interactions

## Interview Questions
1. What's the difference between feature selection and feature extraction?
2. How would you create features from timestamp data?
3. Why is feature engineering considered more impactful than model choice?

## Production Mapping
Feature transforms live in `feature_store/transforms.py`. The FeatureBuilder in `feature_store/builders.py` computes features in parallel via asyncio.gather.


In [ ]:
import pandas as pd, numpy as np
np.random.seed(42)
n = 300
df = pd.DataFrame({
    'capacity_mw': np.random.exponential(500, n),
    'throughput_tons': np.random.exponential(10000, n),
    'latitude': np.random.uniform(-60, 70, n),
    'longitude': np.random.uniform(-180, 180, n),
    'region_risk': np.random.uniform(0, 1, n),
    'age_years': np.random.exponential(30, n),
})
df.head()

In [ ]:
# Ratio feature
df['efficiency_ratio'] = df['throughput_tons'] / (df['capacity_mw'] + 1)
print("Ratio feature (throughput / capacity):")
print(df['efficiency_ratio'].describe())

In [ ]:
# Haversine distance to chokepoints (same as GeospatialTransform)
import math
chokepoints = {
    'Strait_of_Hormuz': (26.5, 56.0),
    'Strait_of_Malacca': (2.0, 102.0),
    'Suez_Canal': (30.5, 32.5),
    'Panama_Canal': (9.0, -79.5),
    'Bab_el_Mandeb': (12.5, 43.5),
}
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))

for name, (lat, lon) in chokepoints.items():
    df[f'dist_{name}'] = df.apply(lambda r: haversine(r['latitude'], r['longitude'], lat, lon), axis=1)
print("Distance to chokepoints (km):")
print(df[[c for c in df.columns if 'dist_' in c]].describe())

In [ ]:
# Polynomial features
from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(df[['capacity_mw', 'age_years']])
print(f"Original features: 2, Polynomial features: {X_poly.shape[1]}")
print(f"Feature names: {poly.get_feature_names_out(['capacity_mw', 'age_years'])}")

In [ ]:
# Binning age into categories
df['age_bin'] = pd.cut(df['age_years'], bins=[0, 10, 30, 50, 200], labels=['new', 'mid', 'old', 'very_old'])
print(df['age_bin'].value_counts())

## Key Takeaways
- Domain knowledge is the best feature engineering guide
- Ratio and aggregate features capture relative behavior
- Geospatial distance features encode supply chain risk
- Polynomial features capture interactions
- All feature engineering must be reproducible (versioned transforms)
- In production, transforms are registered in the Feature Store